In [ ]:
from datetime import datetime
import pandas as pd
from matplotlib import pyplot as plt
import pycountry
from os import listdir as ls
import numpy as np
from IPython.display import display, Markdown
import pycountry_convert as pc
from matplotlib_inline.backend_inline import set_matplotlib_formats
import pycountry

from emu_renewal.constants import OUTPUTS_PATH, DATA_PATH, MOB_COLOURS
from emu_renewal.plotting import MOB_SOURCE_MAP, compare_proc_mob, MOB_ANALYSIS_MAP, MOB_NAME_MAP, CONT_CMAP
from emu_renewal.utils import get_countries_by_continent, get_mob_avail_countries
from emu_renewal.inputs import get_income_group

set_matplotlib_formats("svg")

In [ ]:
job_path = OUTPUTS_PATH / "46076690"
all_countries = ls(job_path)
countries_by_cont = get_countries_by_continent(all_countries)
notes = {
    "g_mob": "Mobility values presented as Google data divided by 100, plus one.",
    "fb_visited_mob": "Mobility values presented as one plus Facebook data.",
    "fb_singletile_mob": "Mobility values presented as one minus Facebook data.",
}

In [ ]:
get_income_group(pycountry.countries.lookup("Pakistan").alpha_3)

Good correspondence:
Mexico - retail and recreation
Czechia - transit stations

Secular trend:
Cote d'Ivoire - grocery and pharmacy


Retail and rec - probably best correlations, large secular trends in African countries, but suggestion of underlying relationship in some (Zimbabwe), close correspondence in some European countries (e.g. Ireland), also close match in some upper middle and high-income North American countries (Mexico)

Grocery and pharmacy - similar trends to retail and rec, but less pronounced, still strong secular trends in many African countries and middle-income countries of Asia, close matches in some European countries (France)
Parks - major fluctuations ranging from directly (Israel, Mexico) to inversely (Ukraine) correlated with var process, but usually apparently uncorrelated (Canada)
Transit stations - correlated with var process (Czechia)
Workplace - similar trends to retail and rec, but less pronounced
Residential - note that it is different calculation, often less variation, suggestion of an inverse relationship in some countries (Malaysia, Greece, Zimbabwe)

FB tiles visited - less underlying secular trend in African countries, close match in some countries (Ireland, Greece, UK)
FB single tile - less variation for African countries, but similar patterns (Greece)

Retail rec - Ireland, Mexico, Kenya, Zimbabwe
Grocery pharm - Mexico (probable relationship), Peru (possible relationship), Spain (relationship changes over time), Cameroon
Parks - South Africa (no clear relationship), Puerto Rico (apparent positive correspondence), Russian Federation (apparent negative relationship)
Transit stations - France (clear correspondence), Turkey (apparent positive relationship), Rwanda (possible relationship in some African countries), Colombia (secular trend in South American countries with possible underlying relationship)
Workplaces - Greece (close relationship), Zimbabwe (close relationship, Uruguay (little relationship in South American countries)
Residential - Finland (inverse relationship in many European countrie), Spain (most notable change early in 2020), Zimbabwe (apparently negative relationship), Afghanistan (very little variation in some African and Asian countries)
FB tiles visited - Czechia (close relationship), Cameroon (less secular trend, but little variation in African countries), Jordan (no obvious correspondence), Latvia (possible relationship with lag - also Canada)
FB single tile - Portugal (stronger correspondence in early epidemic), Malawi (even less variation than FB tiles visited), Madagascar (possible correspondence, but less variation), French Guiana (no obvious correspondence)

In [ ]:
from emu_renewal.inputs import (
    get_google_mobility,
    get_fb_visited_mobility,
    get_fb_singletile_mobility,
)
from emu_renewal.plotting import G_MOB_DOMAIN_CMAP

In [ ]:
fig, axes = plt.subplots(4, 8, figsize=(18, 8))
flat_axes = axes.ravel()
plt.close()
fig.tight_layout()
mob_source = "g_mob"
iso3 = "GAB"
proc_samples = pd.read_hdf(job_path / "GAB" / "no_mob/spaghetti.h5")["process"]
centiles = proc_samples.quantile([0.05, 0.5, 0.95], axis=1).T
flat_axes[0].plot(centiles.index, centiles[0.5], label="process", color="navy")
flat_axes[0].fill_between(centiles.index, centiles[0.05], centiles[0.95], alpha=0.2, color="navy")
if mob_source == "g_mob":
    mob = get_google_mobility(iso3)[mob_type]
elif mob_source == "fb_visited_mob":
    mob = get_fb_visited_mobility(iso3)
elif mob_source == "fb_singletile_mob":
    mob = get_fb_singletile_mobility(iso3)
mobility = mob.loc[(centiles.index[0] < mob.index) & (mob.index < centiles.index[-1])]
mob_name = MOB_NAME_MAP[mob_type]
smoothed_mob = mobility.rolling(7, center=True).mean().dropna()
colour = G_MOB_DOMAIN_CMAP[mob_type] if mob_source == "g_mob" else MOB_COLOURS[mob_type]
flat_axes[0].plot(smoothed_mob.index, smoothed_mob, color=colour)


In [ ]:
fig

# 